# tse-tick — NEEDS extraction & **evaluation** notebook

A **standalone, re-usable acceptance test** for the [`tse-tick`](https://pypi.org/project/tse-tick/)
package, run against a local NEEDS delivery. It exercises every documented access pattern from the
README, **strictly using the package's documented API**, and prints a PASS / FAIL / SKIP verdict.

**How to use:** edit the **CONFIG** cell (your `DATA_ROOT` and the days/tickers your delivery contains),
then *Run All*. Each data-type cell is independently runnable; the final **SUMMARY** cell aggregates the
results. Re-run any section after a `pip install -U tse-tick` to re-test a new release.

**Why it exists:** ad-hoc tests drift into non-standard usage that trips edge paths. This notebook pins
the *one* documented way to use each feature, so a run measures the package, not the caller's improvisation.

### The documented format this notebook holds itself to (from the README)
- **One-shot** — `read_ticks(source, data_type=…, ticker_filter={"7203"}, date="YYYYMMDD", start_time=…, end_time=…)`.
  `source` is a `.zip`, a flat folder, or **any folder above the data** (located by type + date).
- **Two-stage** — `ingest_period(source, store, period, data_type, ticker_filter={…})` → then
  `query_ticks(store, data_type=…, ticker=…, date=…, start_time=…, end_time=…)` (sub-second; adds a `date` column).
- **`ticker_filter` is a *set* of codes** (`{"7203"}`); `date` is `"YYYYMMDD"` / `"YYYYMM"` / `"YYYY"` / a range.
- **Project with Polars `.select(...)` after the read** (not a `columns=` argument).
- **Summary types are date-only** — passing `start_time`/`end_time` raises by design.
- **Discovery** — `get_available_dates` / `get_available_tickers` operate on a built store.

| data_type | subfolder | identifier | intraday time filter |
|---|---|---|---|
| `individual_stock` | `TICST120` (daily, multi-part) | Toyota `7203` | yes |
| `stock_summary` | `TICSS110` (monthly) | Toyota `7203` | no (daily aggregate) |
| `indices` | `TICIT110` / `TICIT010` (2016) | Nikkei 225 `101` | yes |
| `indices_summary` | `TICIS110` / `TICIS010` (2016) | Nikkei 225 `101` | no (daily aggregate) |

> `DATA_ROOT` is treated **read-only**. The notebook writes small test stores under `./nb_eval_stores/`, never under it.

### New in 0.11.6 — part-pruning + one-call two-stage

- **`read_ticks` is part-pruned** for `individual_stock` + a `ticker_filter`: it opens only the contiguous run of numbered parts that hold the ticker (plus the day's trailing off-auction appendix part), not every part of the day. Results are **row-for-row identical** to a full scan; pass `prune_parts=False` to force a full scan. The one-shot checks below therefore also exercise pruning.
- **`extract_to_store(input_root, store, period, ticker, ...)`** does the two-stage path (ingest → query) in one call and is the recommended way to read a ticker you'll query more than once. `tse-tick export --store <dir>` exposes it on the CLI.


In [ ]:
# === Install / upgrade tse-tick to the LATEST release, into THIS kernel ===
# Works whether or not an older version is already installed, and loads the new
# files immediately so you do NOT need to restart the kernel: pip upgrades files on
# disk, but a kernel that already imported tse_tick keeps the OLD version in memory
# until those modules are dropped (the sys.modules purge below). Re-run any time.
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", "tse-tick[query]"],
    check=True,
)
for _m in [m for m in list(sys.modules) if m == "tse_tick" or m.startswith("tse_tick.")]:
    del sys.modules[_m]   # drop the in-memory old version so the next import is fresh
import importlib.metadata, tse_tick
print("tse-tick on disk :", importlib.metadata.version("tse-tick"))
print("imported version :", tse_tick.__version__)

## CONFIG — edit me

In [ ]:
import os

# Read-only root: the parent of the 個別株式{YYYY} year folders.
DATA_ROOT = os.environ.get("TSE_TICK_DATA_ROOT", r"/path/to/NEEDS")  # <- set to your NEEDS root (parent of the 個別株式{YYYY} folders)

# Where this notebook builds its small Parquet test stores (NOT under DATA_ROOT).
STORE_ROOT = os.path.join(os.getcwd(), "nb_eval_stores")

# One evaluation case per (data_type, era). Edit `day` / `period` / `ticker` to days your
# delivery actually contains. `sub` is the type subfolder under 個別株式{year}. `period` is what
# ingest builds (a day for daily individual_stock, a month "YYYYMM" for the monthly types);
# `day` is the single day both access paths are checked on.
CASES = [
    dict(key="indiv2016", label="individual_stock 2016", data_type="individual_stock",
         year=2016, sub="TICST120", period="20160901", day="20160901",
         ticker="7203", time=("09:00:00", "15:00:00")),
    dict(key="ss2023", label="stock_summary 2023", data_type="stock_summary",
         year=2023, sub="TICSS110", period="202305", day="20230508",
         ticker="7203", time=None),
    dict(key="idx2023", label="indices 2023", data_type="indices",
         year=2023, sub="TICIT110", period="202305", day="20230508",
         ticker="101", time=("09:00:00", "15:00:00")),
    dict(key="idx2016", label="indices 2016 (legacy 010)", data_type="indices",
         year=2016, sub="TICIT010", period="201609", day="20160901",
         ticker="101", time=("09:00:00", "15:00:00")),
    dict(key="is2023", label="indices_summary 2023", data_type="indices_summary",
         year=2023, sub="TICIS110", period="202305", day="20230508",
         ticker="101", time=None),
    dict(key="is2016", label="indices_summary 2016 (legacy 010)", data_type="indices_summary",
         year=2016, sub="TICIS010", period="201609", day="20160901",
         ticker="101", time=None),
]

# A non-trading day (Golden Week) for the no-data behaviour check.
HOLIDAY = dict(data_type="indices", year=2023, sub="TICIT110", day="20230504", ticker="101")

## SETUP — run once (imports, helpers, result accumulator)

In [ ]:
import sys, platform, warnings, shutil
os.environ.setdefault("PYTHONUTF8", "1")          # safe non-ASCII printing on Windows consoles
import tse_tick
import polars as pl

import importlib.metadata as _md
if tse_tick.__version__ != _md.version("tse-tick"):
    raise RuntimeError(
        f"Kernel has tse_tick {tse_tick.__version__} but {_md.version('tse-tick')} is installed "
        f"on disk — re-run the install cell (or Kernel -> Restart), then Run All."
    )

print("tse_tick :", tse_tick.get_version())
print("python   :", sys.version.split()[0])
print("OS       :", platform.platform())
print("supported:", tse_tick.get_supported_data_types())
print("years    :", tse_tick.get_supported_years())
print("DATA_ROOT:", DATA_ROOT, "(exists)" if os.path.isdir(DATA_ROOT) else "(MISSING — edit CONFIG)")
os.makedirs(STORE_ROOT, exist_ok=True)

CASE = {c["key"]: c for c in CASES}

# Documented OUTPUT column counts (the store path adds one `date` partition column).
EXPECTED_COLS = {"individual_stock": 95, "stock_summary": 82, "indices": 10, "indices_summary": 17}
# A measure column that must be numeric (Float64) per the README's dtype guarantee.
NUMERIC_PROBE = {"individual_stock": "Execution Price", "stock_summary": "Daily VWAP",
                 "indices": "Index Value", "indices_summary": "AM Opening Price"}
# The in-file identifier column for each type.
CODE_COL = {"individual_stock": "Stock Code", "stock_summary": "Stock Code",
            "indices": "Index Code", "indices_summary": "Index Code"}

RESULTS = []  # (name, status, detail)


class _NoData(Exception):
    """Raised when the data for a case isn't on disk -> SKIP, not FAIL."""


def _add(name, status, detail=""):
    RESULTS.append((name, status, detail))
    mark = {"PASS": "[PASS]", "FAIL": "[FAIL]", "SKIP": "[skip]"}[status]
    print(f"  {mark} {name}" + (f" — {detail}" if detail else ""))


def _check(name, fn):
    try:
        ok, detail = fn()
        _add(name, "PASS" if ok else "FAIL", detail)
    except _NoData as e:
        _add(name, "SKIP", str(e))
    except Exception as e:  # any error using the package is a FAIL of the check
        _add(name, "FAIL", f"{type(e).__name__}: {e}")


def _type_dir(c):
    return os.path.join(DATA_ROOT, f"個別株式{c['year']}", c["sub"])


def _flags(pairs):
    return ", ".join(f"{k}={'ok' if v else 'BAD'}" for k, v in pairs)


def evaluate_case(c):
    """Run the documented one-shot + two-stage paths for one case and record 3 checks."""
    dt, src, ticker, day, tw = c["data_type"], _type_dir(c), c["ticker"], c["day"], c["time"]
    print(f"\n### {c['label']}  (data_type={dt!r}, ticker={ticker}, day={day})")
    if not os.path.isdir(src):
        for suffix in ("one-shot read_ticks", "ingest -> query_ticks", "one-shot == store"):
            _add(f"{c['label']} · {suffix}", "SKIP", f"missing folder {src}")
        return

    captured = {}

    # --- 1) one-shot read_ticks (the documented exploration path) ---
    def _one_shot():
        kw = dict(data_type=dt, ticker_filter={ticker}, date=day)
        if tw:
            kw.update(start_time=tw[0], end_time=tw[1])
        df = tse_tick.read_ticks(src, **kw)
        captured["one"] = df
        if df.height == 0:
            raise _NoData(f"0 rows for {ticker} on {day} — is that day in your data?")
        pairs = [
            ("cols", df.width == EXPECTED_COLS[dt]),
            ("filter", set(df[CODE_COL[dt]].cast(pl.String).str.strip_chars()
                           .str.slice(0, 4).unique().to_list()) == {ticker}),
            (f"{NUMERIC_PROBE[dt]}:Float64", df.schema[NUMERIC_PROBE[dt]] == pl.Float64),
        ]
        if tw:  # non-blank Execution Time must fall inside the window
            et = df["Execution Time"].cast(pl.String).str.strip_chars()
            nb = et.filter(et != "")
            lo, hi = tw[0].replace(":", ""), tw[1].replace(":", "")
            pairs.append(("time-window", nb.len() == 0 or (nb.min() >= lo and nb.max() <= hi)))
        return all(v for _, v in pairs), f"shape={df.shape}; " + _flags(pairs)
    _check(f"{c['label']} · one-shot read_ticks", _one_shot)

    # --- 2) two-stage ingest_period -> query_ticks (the documented scale path) ---
    store = os.path.join(STORE_ROOT, c["key"])
    def _two_stage():
        shutil.rmtree(store, ignore_errors=True)   # fresh store so the eval can't read stale data
        tse_tick.ingest_period(src, store, c["period"], dt, ticker_filter={ticker})
        dates = tse_tick.get_available_dates(store, data_type=dt)
        tickers = tse_tick.get_available_tickers(store, data_type=dt)
        kw = dict(data_type=dt, ticker=ticker, date=day)
        if tw:
            kw.update(start_time=tw[0], end_time=tw[1])
        df = tse_tick.query_ticks(store, **kw)
        captured["store"] = df
        pairs = [
            ("cols(+date)", df.width == EXPECTED_COLS[dt] + 1),
            ("nonempty", df.height > 0),
            ("date listed", day in dates),
            ("ticker listed", ticker in tickers),
        ]
        return all(v for _, v in pairs), f"shape={df.shape}; dates={dates[:3]}; tickers={tickers[:5]}"
    _check(f"{c['label']} · ingest -> query_ticks", _two_stage)

    # --- 3) the two paths must agree on row count for identical filters ---
    def _consistency():
        a, b = captured.get("one"), captured.get("store")
        if a is None or b is None:
            raise _NoData("a prior step produced no frame")
        return a.height == b.height, f"one-shot={a.height} vs store={b.height}"
    _check(f"{c['label']} · one-shot == store (row count)", _consistency)

## 1. `individual_stock` — TICST120 (per-stock tick & quotes, intraday)

Daily, multi-part packaging. `read_ticks` opens **every part** of the day, so a single ticker-day takes
tens of seconds. Quote-only book rows have a blank `Execution Time` and are kept via the `Update Time`
fallback (so the time-window check below only asserts on *non-blank* execution times).

In [ ]:
evaluate_case(CASE["indiv2016"])

## 2. `stock_summary` — TICSS110 (per-stock daily summary)

Daily aggregate (date-only). Multiple rows per stock-day are legitimate per-exchange listings.

In [ ]:
evaluate_case(CASE["ss2023"])

## 3. `indices` — TICIT110 (2017+) / TICIT010 (2016), intraday

`tse-tick` auto-detects the 2016 legacy `…010` fixed-width layout from the filename, so the **same**
`data_type="indices"` call works on both eras (the second case below points at the 2016 folder).

In [ ]:
evaluate_case(CASE["idx2023"])

In [ ]:
evaluate_case(CASE["idx2016"])   # 2016 legacy era, auto-detected

## 4. `indices_summary` — TICIS110 (2017+) / TICIS010 (2016)

Daily aggregate (date-only); the 2016 legacy era is again auto-detected.

In [ ]:
evaluate_case(CASE["is2023"])

In [ ]:
evaluate_case(CASE["is2016"])   # 2016 legacy era, auto-detected

## 5. No-data handling (documented behaviour)

A request on a non-trading day must return a **typed-empty** frame (0 rows, full column set) **and** emit a
capturable `tse_tick.NoDataWarning` — never a crash or a schemaless `(0, 0)` frame.

In [ ]:
def _no_data():
    src = os.path.join(DATA_ROOT, f"個別株式{HOLIDAY['year']}", HOLIDAY["sub"])
    if not os.path.isdir(src):
        raise _NoData(f"missing folder {src}")
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        df = tse_tick.read_ticks(src, data_type=HOLIDAY["data_type"],
                                 ticker_filter={HOLIDAY["ticker"]}, date=HOLIDAY["day"])
    typed_empty = df.height == 0 and df.width == EXPECTED_COLS[HOLIDAY["data_type"]]
    warned = any(issubclass(w.category, tse_tick.NoDataWarning) for w in caught)
    return (typed_empty and warned), f"shape={df.shape}; NoDataWarning={warned}"

_check("no-data (holiday) -> typed-empty + NoDataWarning", _no_data)

## SUMMARY — overall verdict

In [ ]:
from collections import Counter

counts = Counter(s for _, s, _ in RESULTS)
bar = "=" * 70
print(bar)
print(f"tse-tick {tse_tick.get_version()} evaluation — "
      f"{counts['PASS']} passed · {counts['FAIL']} failed · {counts['SKIP']} skipped")
print(bar)
for name, status, detail in RESULTS:
    mark = {"PASS": "[PASS]", "FAIL": "[FAIL]", "SKIP": "[skip]"}[status]
    print(f"{mark} {name}")
    if status == "FAIL" and detail:
        print(f"         {detail}")
print(bar)
if counts["FAIL"]:
    print(f"VERDICT: {counts['FAIL']} FAILURE(S) — see [FAIL] rows above.")
elif counts["PASS"]:
    print("VERDICT: ALL CHECKS PASSED (skips are data not present on this machine).")
else:
    print("VERDICT: nothing ran — check DATA_ROOT in CONFIG.")